# Flight price prediction-REGRESSION

In [15]:
import pandas as pd
df = pd.read_csv(r"C:\Users\razih\OneDrive\Desktop\New\flight\Flight_Price.csv")
df.head(2)

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR ? DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897
1,Air India,1/05/2019,Kolkata,Banglore,CCU ? IXR ? BBI ? BLR,05:50,13:15,7h 25m,2 stops,No info,7662


In [16]:
df.isnull().sum()

Airline            0
Date_of_Journey    0
Source             0
Destination        0
Route              1
Dep_Time           0
Arrival_Time       0
Duration           0
Total_Stops        1
Additional_Info    0
Price              0
dtype: int64

In [17]:
df=df.bfill()
#df.head(2)

In [18]:
#splitting date of journey into day-month-year columns

df["day"] = pd.to_datetime(df["Date_of_Journey"],format="%d/%m/%Y").dt.day
df["month"] = pd.to_datetime(df["Date_of_Journey"],format="%d/%m/%Y").dt.month
df["year"] = pd.to_datetime(df["Date_of_Journey"],format="%d/%m/%Y").dt.year
df = df.drop(["Date_of_Journey"],axis=1)
df.head(2)

,Airline,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,day,month,year
0,IndiGo,Banglore,New Delhi,BLR ? DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897,24,3,2019
1,Air India,Kolkata,Banglore,CCU ? IXR ? BBI ? BLR,05:50,13:15,7h 25m,2 stops,No info,7662,1,5,2019


In [19]:
#changing duration to Hours and Minutes

df["Duration_H"] = df["Duration"].str.extract(r'(\d+)h').fillna(0).astype(int)
df["Duration_m"] = df["Duration"].str.extract(r'(\d+)m').fillna(0).astype(int)
df = df.drop(["Duration"],axis=1)
df.head(2)

,Airline,Source,Destination,Route,Dep_Time,Arrival_Time,Total_Stops,Additional_Info,Price,day,month,year,Duration_H,Duration_m
0,IndiGo,Banglore,New Delhi,BLR ? DEL,22:20,01:10 22 Mar,non-stop,No info,3897,24,3,2019,2,50
1,Air India,Kolkata,Banglore,CCU ? IXR ? BBI ? BLR,05:50,13:15,2 stops,No info,7662,1,5,2019,7,25


In [20]:
# Splitting Departure time into Hours and Minutes

df["Dep-H"] = df["Dep_Time"].str.split(":").str[0].astype(int)
df["Dep-m"] = df["Dep_Time"].str.split(":").str[1].astype(int)
df=df.drop(["Dep_Time"],axis=1)
df.head(2)

,Airline,Source,Destination,Route,Arrival_Time,Total_Stops,Additional_Info,Price,day,month,year,Duration_H,Duration_m,Dep-H,Dep-m
0,IndiGo,Banglore,New Delhi,BLR ? DEL,01:10 22 Mar,non-stop,No info,3897,24,3,2019,2,50,22,20
1,Air India,Kolkata,Banglore,CCU ? IXR ? BBI ? BLR,13:15,2 stops,No info,7662,1,5,2019,7,25,5,50


In [21]:
#changing arrival time into hours and minutes
df["Arrival_Time"] = df["Arrival_Time"].str.extract(r"(\d{1,2}:\d{2})")
df["Arrival_H"] = df["Arrival_Time"].str.split(":").str[0].astype(int)
df["Arrival_m"] = df["Arrival_Time"].str.split(":").str[1].astype(int)
df = df.drop(["Arrival_Time"],axis=1)
df.head(2)


,Airline,Source,Destination,Route,Total_Stops,Additional_Info,Price,day,month,year,Duration_H,Duration_m,Dep-H,Dep-m,Arrival_H,Arrival_m
0,IndiGo,Banglore,New Delhi,BLR ? DEL,non-stop,No info,3897,24,3,2019,2,50,22,20,1,10
1,Air India,Kolkata,Banglore,CCU ? IXR ? BBI ? BLR,2 stops,No info,7662,1,5,2019,7,25,5,50,13,15


In [22]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import pandas as pd

cols = ["Airline","Source","Destination","Route","Additional_Info"]

label = OneHotEncoder(sparse_output= False,handle_unknown='ignore')

for col in cols:
    encoded = label.fit_transform(df[[col]])  

    encoded_df = pd.DataFrame(
        encoded,
        columns=label.get_feature_names_out([col])
    )

    df = pd.concat([df, encoded_df], axis=1)
    df = df.drop(col, axis=1)

# Label Encoding for ordinal feature
label = LabelEncoder()
df["Total_Stops"] = label.fit_transform(df["Total_Stops"])

df.head(2)

,Total_Stops,Price,day,month,year,Duration_H,Duration_m,Dep-H,Dep-m,Arrival_H,...,Additional_Info_1 Long layover,Additional_Info_1 Short layover,Additional_Info_2 Long layover,Additional_Info_Business class,Additional_Info_Change airports,Additional_Info_In-flight meal not included,Additional_Info_No Info,Additional_Info_No check-in baggage included,Additional_Info_No info,Additional_Info_Red-eye flight
0,4,3897,24,3,2019,2,50,22,20,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1,7662,1,5,2019,7,25,5,50,13,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [23]:
#importing the algorithms

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Assuming df is already defined and contains the dataset
X = df.drop(["Price"], axis=1)
y = df["Price"]

# Splitting dataset into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# List of models to evaluate
models = [
    LinearRegression(),
    DecisionTreeRegressor(),
    RandomForestRegressor(),
    GradientBoostingRegressor(),
    XGBRegressor(),
]

# Training and evaluating models
for model in models:
    model.fit(x_train, y_train)
    train_prediction = model.predict(x_train)
    test_prediction = model.predict(x_test)
    
    print("\n")
    print(f"{type(model).__name__}")
    print("*Train*")
    print(f"MSE: {mean_squared_error(y_train, train_prediction)}")
    print(f"R2: {r2_score(y_train, train_prediction)}")
    print("\n")
    print("*Test*")
    print(f"MSE: {mean_squared_error(y_test, test_prediction)}")
    print(f"R2: {r2_score(y_test, test_prediction)}")




LinearRegression
*Train*
MSE: 5287539.763570019
R2: 0.7516020698509418


*Test*
MSE: 5395654.541310307
R2: 0.7450546590787883


DecisionTreeRegressor
*Train*
MSE: 94385.73389109915
R2: 0.9955659490079524


*Test*
MSE: 3553138.2950137784
R2: 0.8321137784068514


RandomForestRegressor
*Train*
MSE: 360274.35602925887
R2: 0.9830750389926088


*Test*
MSE: 2716249.4313490405
R2: 0.8716568801800717


GradientBoostingRegressor
*Train*
MSE: 3632491.7379168426
R2: 0.8293528806726391


*Test*
MSE: 3822702.563260128
R2: 0.8193768335668741


XGBRegressor
*Train*
MSE: 1107120.875
R2: 0.9479897022247314


*Test*
MSE: 2004553.375
R2: 0.905284583568573


In [25]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split


# Assuming df is already defined and contains the dataset
X = df.drop(["Price"], axis=1)
y = df["Price"]

# Splitting dataset into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# List of models to evaluate
models = [
    LinearRegression(),
    DecisionTreeRegressor(),
    RandomForestRegressor(),
    GradientBoostingRegressor(),
    XGBRegressor(),
]

mlflow.set_experiment("Regression_Flight_price_prediction")
mlflow.set_tracking_uri("http://127.0.0.1:5000")
# Training and evaluating models with MLflow tracking
for model in models:
    with mlflow.start_run():
        model.fit(x_train, y_train)
        train_prediction = model.predict(x_train)
        test_prediction = model.predict(x_test)
        
        train_mse = mean_squared_error(y_train, train_prediction)
        train_r2 = r2_score(y_train, train_prediction)
        test_mse = mean_squared_error(y_test, test_prediction)
        test_r2 = r2_score(y_test, test_prediction)
        
        mlflow.log_param("model_name", type(model).__name__)
        mlflow.log_metric("train_mse", train_mse)
        mlflow.log_metric("train_r2", train_r2)
        mlflow.log_metric("test_mse", test_mse)
        mlflow.log_metric("test_r2", test_r2)
        mlflow.sklearn.log_model(model, type(model).__name__)
        
        print("\n")
        print(f"{type(model).__name__}")
        print("*Train*")
        print(f"MSE: {train_mse}")
        print(f"R2: {train_r2}")
        print("\n")
        print("*Test*")
        print(f"MSE: {test_mse}")
        print(f"R2: {test_r2}")


2026/05/07 13:43:32 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.




LinearRegression
*Train*
MSE: 5287539.763570019
R2: 0.7516020698509418


*Test*
MSE: 5395654.541310307
R2: 0.7450546590787883
🏃 View run classy-moose-260 at: http://127.0.0.1:5000/#/experiments/458478889590703516/runs/b3dc94013be44450af2882ebb586bc29
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/458478889590703516


2026/05/07 13:43:39 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.




DecisionTreeRegressor
*Train*
MSE: 94385.73389109915
R2: 0.9955659490079524


*Test*
MSE: 3325065.1912728124
R2: 0.8428902606191613
🏃 View run fearless-dove-74 at: http://127.0.0.1:5000/#/experiments/458478889590703516/runs/032348782cc94646b79e652c6ec286e9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/458478889590703516


2026/05/07 13:43:52 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.




RandomForestRegressor
*Train*
MSE: 347694.08214216976
R2: 0.9836660348307472


*Test*
MSE: 2780022.9116447996
R2: 0.8686435753898458
🏃 View run gregarious-mole-676 at: http://127.0.0.1:5000/#/experiments/458478889590703516/runs/0ee40f003f4743148a1ad4dae1274c7c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/458478889590703516


2026/05/07 13:44:01 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.




GradientBoostingRegressor
*Train*
MSE: 3632491.7379168435
R2: 0.829352880672639


*Test*
MSE: 3820406.5820612293
R2: 0.8194853189610039
🏃 View run trusting-hog-477 at: http://127.0.0.1:5000/#/experiments/458478889590703516/runs/4a7cb03e001a4f92b0cae2cce2d9901f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/458478889590703516


2026/05/07 13:44:08 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.




XGBRegressor
*Train*
MSE: 1107120.875
R2: 0.9479897022247314


*Test*
MSE: 2004553.375
R2: 0.905284583568573
🏃 View run bold-shad-458 at: http://127.0.0.1:5000/#/experiments/458478889590703516/runs/2e99dc484e2b4f208439b92d6c16b675
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/458478889590703516
